In [1]:
import librosa
import librosa.display
from IPython.display import Audio
import numpy as np
import pandas as pd
import os

ImportError: Matplotlib requires numpy>=1.23; you have 1.21.6

In [3]:
# LOAD IN FILE
# main_dir = 'D:/Makaleler/makale19 speech recog/EMOVO/'
main_dir = "C:/Users/Aruay/Desktop/ra application/project/dataset/EMOVO"
#main_dir ='D:/Makaleler/makale19 speech recog/ravdes/ex/'
sub_dir = os.listdir(main_dir)
x, sr = librosa.load(main_dir+'/'+sub_dir[2]+'/dis-f2-b1.wav')
# PLAY any AUDIO FILE
#librosa.output.write_wav('MaleNeutral.wav', x, sr)
Audio(data=x, rate=sr)

ModuleNotFoundError: No module named 'numba'

In [10]:
import soundfile as sf

file_path = main_dir + '/' + sub_dir[2] + '/dis-f2-b1.wav'
x, sr = sf.read(file_path)  # Load the audio using soundfile directly
print(f"Sample Rate: {sr}, Audio Data Shape: {x.shape}")

Sample Rate: 48000, Audio Data Shape: (118784, 2)


In [14]:
def extract_feature(file_name, offst=0.5):
    #X, sample_rate = librosa.load(file_name, res_type='kaiser_fast',offset=offst)
    x, sample_rate = sf.read(file_name)
    start_sample = int(offst * sample_rate)
    X = x[start_sample:]
    
    stft = np.abs(librosa.stft(X))
    chroma_cq = librosa.feature.chroma_cqt(y=X, sr=sample_rate)
    cqt=np.mean(chroma_cq,axis=1)
    
    #chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate,n_fft=2048).T,axis=0)
    #chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T,axis=0)
    
    mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T,axis=0)
    
    mel = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate,n_mels=128,fmax=8000).T,axis=0)
    #mel = np.mean(librosa.feature.melspectrogram(X, sr=sample_rate).T,axis=0)
    
    contrast = np.mean(librosa.feature.spectral_contrast(S=stft, sr=sample_rate).T,axis=0)
    
    tonnetz = np.mean(librosa.feature.tonnetz(y=librosa.effects.harmonic(X), sr=sample_rate).T,axis=0)
    return mfccs,cqt,mel,contrast,tonnetz

def extract_featurev2(file_name):
    X, sample_rate = librosa.load(file_name, res_type='kaiser_fast', sr=None)   
    #mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=64).T, axis=0)
    C = np.abs(librosa.cqt(X, sr=sr, fmin=librosa.note_to_hz('C2'), n_bins=60))
    #d=librosa.amplitude_to_db(C)
    d=librosa.power_to_db(C)
    cqt = np.mean(d,axis=1)
    return cqt

In [12]:
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)

In [15]:
a= []
b= []
c= []
d= []
e= []

#from warnings import warn
import warnings
from pathlib import Path

# run block of code and catch warnings
with warnings.catch_warnings():
	# ignore all caught warnings
	warnings.filterwarnings("ignore")
	# execute code that will generate warnings

emotion = []
path = []
gender = []

X4= pd.DataFrame()

n=0
for root, dirs, files in os.walk(main_dir, topdown=False):
    for f in files:
        if(root[-2]=='f') or (root[-2]=='m'):
            #print(os.path.join(root,"   ;   ", f),n)
            #print(os.path.join(root, f))
            file_path = Path(root) / f
            a,b,c,d,e=extract_feature(file_path)
            #a=extract_feature(main_dir + i + '/' + f)
            tot= []
            for x in a:tot.append(x)
            for x in b:tot.append(x)
            for x in c:tot.append(x)
            for x in d:tot.append(x)
            for x in e:tot.append(x)
            X4[n]=np.asarray(tot)
            n+=1;
            #print(tot)

SystemError: initialization of _internal failed without raising an exception

In [19]:
len(a),len(b),len(c),len(d),len(e),

(40, 12, 128, 7, 6)

In [20]:
X6= pd.DataFrame(X4.T)

In [21]:
type(X6), X6.shape

(pandas.core.frame.DataFrame, (490, 193))

In [22]:
def decompose_emovo():
        #EMODB_PATH = '/kaggle/input/berlin-database-of-emotional-speech-emodb/wav'
        emotion = []
        path = []
        gender = []  
    
        for root, dirs, files in os.walk(main_dir):
            for name in files:
                if(root[-2]=='f') or (root[-2]=='m'):
                    #print(name[0:3])
                    if name[0:3]=='dis':emotion.append(1)
                    elif name[0:3]=='gio':emotion.append(2)
                    elif name[0:3]=='neu':emotion.append(3)
                    elif name[0:3]=='pau':emotion.append(4)
                    elif name[0:3]=='rab':emotion.append(5)
                    elif name[0:3]=='sor':emotion.append(6)
                    elif name[0:3]=='tri':emotion.append(7)
                    else: emotion.append(0)

                    if name[4]=='f':gender.append(1)
                    elif name[4]=='m':gender.append(2)
                    else: gender.append(0)
                    path.append(os.path.join(  name)) #root ,

        emodb_df = pd.DataFrame(emotion, columns=['labels'])
        emodb_df['source'] = 'EMOVO'
        gender_df = pd.DataFrame(gender, columns=['gender'])
        emodb_df = pd.concat([emodb_df, gender_df, pd.DataFrame(path, columns=['path'])], axis=1)
        
        return emodb_df 

In [23]:
df=decompose_emovo()

In [24]:
df

,labels,source,gender,path
0,1,EMOVO,1,dis-f2-b1.wav
1,1,EMOVO,1,dis-f2-b2.wav
2,1,EMOVO,1,dis-f2-b3.wav
3,1,EMOVO,1,dis-f2-d1.wav
4,1,EMOVO,1,dis-f2-d2.wav
...,...,...,...,...
485,7,EMOVO,2,tri-m3-n1.wav
486,7,EMOVO,2,tri-m3-n2.wav
487,7,EMOVO,2,tri-m3-n3.wav
488,7,EMOVO,2,tri-m3-n4.wav


In [25]:
df[df['labels']==0]

,labels,source,gender,path


In [26]:
df[df['gender']==0]

,labels,source,gender,path


In [27]:
X6

,0,1,2,3,4,5,6,7,8,9,...,183,184,185,186,187,188,189,190,191,192
0,-345.518219,68.940361,-17.371601,20.530304,-6.694086,-13.612593,-19.310644,6.167040,-9.788841,1.011834,...,19.359426,16.796542,16.864012,35.886788,0.001853,0.004747,-0.011141,0.088005,-0.008819,0.034074
1,-330.795441,63.394062,-10.537325,31.389908,-5.246548,-9.138075,-17.628042,3.866386,-15.923525,-2.911236,...,17.852244,18.602973,18.137959,38.004999,0.064404,-0.028997,0.009378,0.059026,-0.028268,0.028059
2,-319.857910,91.452522,-17.585133,22.173920,-5.645752,-9.291864,-17.763767,11.816550,-10.021160,2.564129,...,19.627506,16.708205,17.350792,38.718178,0.053473,-0.054322,-0.031217,-0.015067,-0.019440,0.006318
3,-337.973419,73.223236,-20.652866,18.710974,-7.758951,-6.323913,-23.485775,2.914072,-9.011644,4.759130,...,18.739718,17.683649,16.514991,36.121839,-0.053672,0.036107,-0.002300,0.043943,0.002524,0.010426
4,-318.662109,83.108940,-22.048302,26.141157,-11.589247,-13.105200,-18.586666,3.807863,-16.361599,-1.191410,...,20.021857,18.177759,18.522894,37.018479,0.017552,0.021058,0.038102,-0.002861,-0.025637,0.016537
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,-427.713867,126.392067,16.361639,24.903133,-21.689814,9.562323,9.067567,-1.878355,-13.065454,10.922304,...,16.536089,15.881731,18.443498,40.144173,-0.017802,0.023098,0.130219,0.076275,-0.008191,-0.029671
486,-432.355957,97.083054,25.271561,34.355991,-29.937775,10.331720,12.702005,-1.664956,-12.328322,6.984700,...,16.113273,16.514157,18.800086,40.849677,-0.027998,-0.025539,-0.003622,0.090199,-0.020974,0.000534
487,-457.518585,102.663628,16.415854,35.759136,-9.032775,9.086905,3.213672,-4.287372,-6.855869,11.775465,...,14.285393,16.011231,18.473241,40.987945,0.088767,0.065154,0.170505,-0.067772,0.058331,-0.019435
488,-522.311584,86.625275,20.204185,41.819397,-7.047180,14.859884,5.611393,3.489742,-4.076044,11.922163,...,14.503361,15.854401,17.343523,40.235118,0.031831,-0.014209,0.000703,-0.059313,0.022512,0.021434


In [28]:
X7=pd.concat([X6,df['gender'],df['labels']],axis=1)
#X8=pd.concat([X8,pd.DataFrame(gender),pd.DataFrame(emotion)],axis=1)

In [29]:
X7.head()

,0,1,2,3,4,5,6,7,8,9,...,185,186,187,188,189,190,191,192,gender,labels
0,-345.518219,68.940361,-17.371601,20.530304,-6.694086,-13.612593,-19.310644,6.167040,-9.788841,1.011834,...,16.864012,35.886788,0.001853,0.004747,-0.011141,0.088005,-0.008819,0.034074,1,1
1,-330.795441,63.394062,-10.537325,31.389908,-5.246548,-9.138075,-17.628042,3.866386,-15.923525,-2.911236,...,18.137959,38.004999,0.064404,-0.028997,0.009378,0.059026,-0.028268,0.028059,1,1
2,-319.857910,91.452522,-17.585133,22.173920,-5.645752,-9.291864,-17.763767,11.816550,-10.021160,2.564129,...,17.350792,38.718178,0.053473,-0.054322,-0.031217,-0.015067,-0.019440,0.006318,1,1
3,-337.973419,73.223236,-20.652866,18.710974,-7.758951,-6.323913,-23.485775,2.914072,-9.011644,4.759130,...,16.514991,36.121839,-0.053672,0.036107,-0.002300,0.043943,0.002524,0.010426,1,1
4,-318.662109,83.108940,-22.048302,26.141157,-11.589247,-13.105200,-18.586666,3.807863,-16.361599,-1.191410,...,18.522894,37.018479,0.017552,0.021058,0.038102,-0.002861,-0.025637,0.016537,1,1


In [30]:
X7.tail()

,0,1,2,3,4,5,6,7,8,9,...,185,186,187,188,189,190,191,192,gender,labels
485,-427.713867,126.392067,16.361639,24.903133,-21.689814,9.562323,9.067567,-1.878355,-13.065454,10.922304,...,18.443498,40.144173,-0.017802,0.023098,0.130219,0.076275,-0.008191,-0.029671,2,7
486,-432.355957,97.083054,25.271561,34.355991,-29.937775,10.331720,12.702005,-1.664956,-12.328322,6.984700,...,18.800086,40.849677,-0.027998,-0.025539,-0.003622,0.090199,-0.020974,0.000534,2,7
487,-457.518585,102.663628,16.415854,35.759136,-9.032775,9.086905,3.213672,-4.287372,-6.855869,11.775465,...,18.473241,40.987945,0.088767,0.065154,0.170505,-0.067772,0.058331,-0.019435,2,7
488,-522.311584,86.625275,20.204185,41.819397,-7.047180,14.859884,5.611393,3.489742,-4.076044,11.922163,...,17.343523,40.235118,0.031831,-0.014209,0.000703,-0.059313,0.022512,0.021434,2,7
489,-523.225403,94.773750,26.561371,47.559616,-3.906190,4.537993,3.212156,3.126236,-5.446962,10.616609,...,18.290507,40.380914,0.006763,-0.004660,-0.009122,-0.008596,0.008808,0.005311,2,7


In [31]:
X7.to_csv("featureEMOVO.csv",index=False,header=True)
#audio_df.to_csv("audioELM.csv",index=False,header=True)

In [83]:
dir ='D:/codes/datasets/ravdes/'
X7.to_excel(dir+"featureEMOVO.xlsx",index=False) 

In [32]:
import os
data = pd.read_csv('featureEMOVO.csv')

In [33]:
data

,0,1,2,3,4,5,6,7,8,9,...,185,186,187,188,189,190,191,192,gender,labels
0,-345.518219,68.940361,-17.371601,20.530304,-6.694086,-13.612593,-19.310644,6.167040,-9.788841,1.011834,...,16.864012,35.886788,0.001853,0.004747,-0.011141,0.088005,-0.008819,0.034074,1,1
1,-330.795441,63.394062,-10.537325,31.389908,-5.246548,-9.138075,-17.628042,3.866386,-15.923525,-2.911236,...,18.137959,38.004999,0.064404,-0.028997,0.009378,0.059026,-0.028268,0.028059,1,1
2,-319.857910,91.452522,-17.585133,22.173920,-5.645752,-9.291864,-17.763767,11.816550,-10.021160,2.564129,...,17.350792,38.718178,0.053473,-0.054322,-0.031217,-0.015067,-0.019440,0.006318,1,1
3,-337.973419,73.223236,-20.652866,18.710974,-7.758951,-6.323913,-23.485775,2.914072,-9.011644,4.759130,...,16.514991,36.121839,-0.053672,0.036107,-0.002300,0.043943,0.002524,0.010426,1,1
4,-318.662109,83.108940,-22.048302,26.141157,-11.589247,-13.105200,-18.586666,3.807863,-16.361599,-1.191410,...,18.522894,37.018479,0.017552,0.021058,0.038102,-0.002861,-0.025637,0.016537,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,-427.713867,126.392067,16.361639,24.903133,-21.689814,9.562323,9.067567,-1.878355,-13.065454,10.922304,...,18.443498,40.144173,-0.017802,0.023098,0.130219,0.076275,-0.008191,-0.029671,2,7
486,-432.355957,97.083054,25.271561,34.355991,-29.937775,10.331720,12.702005,-1.664956,-12.328322,6.984700,...,18.800086,40.849677,-0.027998,-0.025539,-0.003622,0.090199,-0.020974,0.000534,2,7
487,-457.518585,102.663628,16.415854,35.759136,-9.032775,9.086905,3.213672,-4.287372,-6.855869,11.775465,...,18.473241,40.987945,0.088767,0.065154,0.170505,-0.067772,0.058331,-0.019435,2,7
488,-522.311584,86.625275,20.204185,41.819397,-7.047180,14.859884,5.611393,3.489742,-4.076044,11.922163,...,17.343523,40.235118,0.031831,-0.014209,0.000703,-0.059313,0.022512,0.021434,2,7


In [34]:
data.shape

(490, 195)